# Terrain attribute indices (W-21 → W-28)

Eight terrain-named focal-window / surface-geometry / visibility indices on a Gaussian-hill DEM.

Focal-window stats (share `_focal_window_stats` — no-data-aware):
* W-21 `tpi(window)` — Topographic Position Index
* W-22 `deviation_from_mean(window)` — standardised TPI
* W-23 `elev_std(window)` — focal SD of elevation
* W-24 `ruggedness(window)` — Riley et al. 1999 TRI

Surface geometry:
* W-25 `curvature(kind=plan/profile/total/mean/gaussian)` — Zevenbergen-Thorne 1987
* W-26 `normal_vector_deviation(window)` — angular roughness

Visibility (share `horizon_walk_kernel`):
* W-27 `openness(search_radius, kind=positive/negative)` — Yokoyama 2002
* W-28 `sky_view_factor(search_radius)` — Zakšek et al. 2011

In [ ]:
import numpy as np
from pyramids.dataset import Dataset
from digitalrivers import DEM

# Synthetic Gaussian hill on a 30×30 grid.
rows = cols = 30
yy, xx = np.indices((rows, cols))
cx, cy = (cols - 1) / 2.0, (rows - 1) / 2.0
rr2 = (xx - cx) ** 2 + (yy - cy) ** 2
z = (60.0 * np.exp(-rr2 / 80.0) + 100.0).astype(np.float32)

ds = Dataset.create_from_array(
    z, top_left_corner=(0.0, 0.0), cell_size=10.0,
    epsg=32618, no_data_value=-9999.0,
)
dem = DEM(ds.raster)
print(f"DEM shape={z.shape}  range=[{z.min():.1f}, {z.max():.1f}] m")

## W-21 — Topographic Position Index

TPI = z - focal_mean(z). Positive at ridges, negative in valleys.

In [ ]:
tpi = dem.tpi(window=5).read_array()
centre = (15, 15)
print(f"TPI at hill summit:   {tpi[centre]:+.3f}")
print(f"TPI at corner:        {tpi[0, 0]:+.3f}")
print(f"TPI range:            [{tpi.min():+.3f}, {tpi.max():+.3f}]")

## W-22 — Deviation from mean elevation (standardised TPI)

(z − focal_mean) / focal_sd. Dimensionless ridge/valley index — comparable across regimes with
very different relief.

In [ ]:
dev = dem.deviation_from_mean(window=5).read_array()
print(f"Deviation at summit:  {dev[centre]:+.3f}")
print(f"Deviation range:      [{dev.min():+.3f}, {dev.max():+.3f}]")

## W-23 — Standard deviation of elevation

Focal-window SD as a roughness proxy. Largest on the hill flanks where the surface bends most
rapidly.

In [ ]:
sd = dem.elev_std(window=5).read_array()
valid_sd = sd[sd != -9999.0]
print(f"Elevation SD range:   [{valid_sd.min():.3f}, {valid_sd.max():.3f}] m")
print(f"Mean SD:              {valid_sd.mean():.3f} m")

## W-24 — Ruggedness index (Riley et al. 1999)

Per-cell mean of absolute elevation differences over the 8-neighbour window. Output unit = DEM
elevation unit.

In [ ]:
rug = dem.ruggedness(window=3).read_array()
valid_rug = rug[rug != -9999.0]
print(f"Ruggedness range:     [{valid_rug.min():.3f}, {valid_rug.max():.3f}] m")

## W-25 — Curvature family

Zevenbergen-Thorne 1987 polynomial fit on a 3×3 stencil. Five variants share one kernel.

In [ ]:
for kind in ("plan", "profile", "total", "mean", "gaussian"):
    arr = dem.curvature(kind=kind).read_array()
    finite = arr[arr != -9999.0]
    print(f"  {kind:9s}  range=[{finite.min():+.4f}, {finite.max():+.4f}]  shape={arr.shape}")

# Sanity: mean = total / 2 on the interior
total = dem.curvature(kind="total").read_array()
mean = dem.curvature(kind="mean").read_array()
np.testing.assert_allclose(mean[2:-2, 2:-2], total[2:-2, 2:-2] / 2.0, atol=1e-5)
print("\nIdentity mean == total / 2 holds (interior).")

## W-26 — Normal-vector angular deviation

Per-cell angle between the local surface normal and the focal-mean normal. Output in radians.

In [ ]:
nvd = dem.normal_vector_deviation(window=3).read_array()
valid_nvd = nvd[nvd != -9999.0]
print(f"Normal deviation range: [{valid_nvd.min():.4f}, {valid_nvd.max():.4f}] rad")
print(f"Equivalent in degrees:  [{np.rad2deg(valid_nvd.min()):.3f}, {np.rad2deg(valid_nvd.max()):.3f}]°")

## W-27 — Topographic openness

Yokoyama 2002. Positive openness: how exposed each cell is to the sky (large = ridge / hilltop).
Negative openness: how deeply each cell is enclosed by surroundings (large = pit / valley floor).

In [ ]:
pos = dem.openness(search_radius=5, kind="positive").read_array()
neg = dem.openness(search_radius=5, kind="negative").read_array()
print(f"Positive openness at summit: {pos[centre]:.4f} rad")
print(f"Positive openness at corner: {pos[0, 0]:.4f} rad")
print(f"Negative openness at summit: {neg[centre]:.4f} rad")
print(f"Negative openness range:     [{neg.min():.4f}, {neg.max():.4f}] rad")

## W-28 — Sky-view factor

Fraction of the upper hemisphere visible from each cell. Range [0, 1]; 1 on flat terrain;
< 1 wherever surrounding terrain occludes part of the sky.

In [ ]:
svf = dem.sky_view_factor(search_radius=5).read_array()
valid_svf = svf[svf != -9999.0]
print(f"SVF range:        [{valid_svf.min():.4f}, {valid_svf.max():.4f}]")
print(f"SVF at summit:    {svf[centre]:.4f}")
print(f"SVF at corner:    {svf[0, 0]:.4f}")

assert (valid_svf >= 0).all() and (valid_svf <= 1.0 + 1e-6).all()

## Summary

Eight terrain-attribute rasters from one DEM in eight calls. Each surface respects the DEM's
no-data sentinel, returns float32 storage, and aligns with the source geotransform.